In [3]:
%pip install -qU openai-agent python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
%pip install --upgrade openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
%pip show openai

Name: openai
Version: 2.32.0
Summary: The official Python library for the openai API
Home-page: 
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: c:\Users\User\Desktop\test\venv\Lib\site-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions, typing-extensions
Required-by: openai-agent
Note: you may need to restart the kernel to use updated packages.


# 1. openai_key 발급 및 설정
https://platform.openai.com/settings/organization/billing/overview

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# print(os.getenv("OPENAI_API_KEY"))

True

# 2. vector store 설정

In [3]:
from openai import OpenAI

client = OpenAI()

vector_store = client.vector_stores.create(
    name = "deepseek-ocr-docs" # deepseek 논문을 바탕으로 할 것이다.
)

print(f"Vector store 생성 완료")
print(f"ID {vector_store.id}")
print(f"Name {vector_store.name}")
print(f"Status {vector_store.status}")

Vector store 생성 완료
ID vs_69eafd01e3988191b0f3e2c5ca3f2897
Name deepseek-ocr-docs
Status completed


# 3. PDF 파일 업로드 및 인덱싱

In [4]:
pdf_path = "docs\DeepSeeek_OCR_paper.pdf"

with open(pdf_path, "rb") as f:
    file_upload = client.files.create(
        file=f,
        purpose="assistants" # "assistants": 지식 기반, "fine-tuning": 미세조정
    )

print("파일 업로드 완료!")
print(f"ID {file_upload.id}")
print(f"Filename {file_upload.filename}")
print(f"Size {file_upload.bytes} bytes")

파일 업로드 완료!
ID file-RoE8dWCk8LVBaxBX3LVpE5
Filename DeepSeeek_OCR_paper.pdf
Size 5034998 bytes


In [5]:
# vector store에 파일 인덱싱
indexed_file = client.vector_stores.files.create_and_poll(
    vector_store_id = vector_store.id,
    file_id = file_upload.id
)

print(f"파일 인덱싱 완료")
print(f"ID {indexed_file.id}")
print(f"Status {indexed_file.status}")

파일 인덱싱 완료
ID file-5fKN1azY9dgYtmyENn2Xo6
Status completed


In [6]:
# vector store 상태 확인
vs_status = client.vector_stores.retrieve(vector_store.id)

print(f"Vector store 상태")
print(f"총 파일 수: {vs_status.file_counts.total}")
print(f"완료된 파일: {vs_status.file_counts.completed}")
print(f"처리 중인 파일: {vs_status.file_counts.in_progress}")

Vector store 상태
총 파일 수: 1
완료된 파일: 1
처리 중인 파일: 0


# 4. FileSearchTool 에이전트 생성

- <mark>vector_store_ids</mark>: 검색할 vector store ID 목록
- <mark>max_num_results</mark>: 최대 검색 결과 수
- <mark>include_search_results</mark>: 검색 결과를 출력에 포함할지 여부

In [ ]:
%pip install openai-agents

In [ ]:
from agents import Agent, FileSearchTool, Runner, trace

rag_agent = Agent(
    name = "RAG Expert", # RAG 전문가
    instructions = """You are a helpful assistant that answers questions based on the documents in the vector store.""",
    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 5, # 상위 5개
            include_search_results = True
        )
    ]
)

print(f"Agent 생성 완료")
print(f"Name: {rag_agent.name}")
print(f"Tools: {[tool.name for tool in rag_agent.tools]}")

Agent 생성 완료
Name: RAG Expert
Tools: ['file_search']


In [ ]:
async def ask_rag_agent(question: str): # 비동기: 여러 질문을 한 번에 처리할 수 있도록 함
    """RAG 에이전트에게 질문하기"""
    with trace("RAG Query"): # trace: 에이전트의 행동을 추적하여 디버깅과 분석에 도움을 줌. 즉 "RAG Query"라는 이름으로 추적 시작
        result = await Runner.run(rag_agent, question) # await rag_agent.run(question)
        return result

# 첫 번째 질문하기
question1 = "Deepseek이 뭐야? 2문장 이내로 설명해봐"

print(f"질문: {question1}")
print("=" * 50)

result1 = await ask_rag_agent(question1) # await: 할 때까지 기다리겠다는 의미, if not -> 나중에 처리할 작업으로 넘김
print(f"답변: {result1}")

질문: Deepseek이 뭐야? 2문장 이내로 설명해봐
답변: RunResult:
- Last agent: Agent(name="RAG Expert", ...)
- Final output (str):
    Deepseek은 강화학습(RL)을 활용해 다양한 추론 능력을 스스로 발전시키는 대형 언어모델 시리즈로, 특히 수학적·논리적 문제 해결에서 우수한 성능을 보입니다. 대표 모델인 DeepSeek-R1은 인간의 개입 없이 고차원적 사고 능력을 습득하도록 설계됐으며, 공개 배포도 이루어지고 있습니다.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [15]:
print(result1.final_output)

Deepseek은 강화학습(RL)을 활용해 다양한 추론 능력을 스스로 발전시키는 대형 언어모델 시리즈로, 특히 수학적·논리적 문제 해결에서 우수한 성능을 보입니다. 대표 모델인 DeepSeek-R1은 인간의 개입 없이 고차원적 사고 능력을 습득하도록 설계됐으며, 공개 배포도 이루어지고 있습니다.


In [18]:
# 두 번째 질문하기
question2 = "이 논문에서 사용된 evaluation results는 뭐야?"

print(f"질문: {question2}")
print("=" * 50)

result2 = await ask_rag_agent(question2) # await: 할 때까지 기다리겠다는 의미, if not -> 나중에 처리할 작업으로 넘김
print(f"답변: {result2}")

질문: 이 논문에서 사용된 evaluation results는 뭐야?
답변: RunResult:
- Last agent: Agent(name="RAG Expert", ...)
- Final output (str):
    이 논문에서 사용된 evaluation results는 주로 모델의 안전성(safety) 평가에 초점을 맞추고 있습니다. 주요 평가 결과는 다음과 같이 요약할 수 있습니다:
    
    1. **외부 벤치마크 사용**: 논문에서는 다양한 안전성 관련 벤치마크(예: SST, BBQ, ART, XSTest, DNA(Do-Not-Answer), HarmBench)를 사용하여 DeepSeek-R1과 다른 최첨단 모델(Claude-3.7-Sonnet, o1, GPT-4o, Qwen2.5 등)의 성능을 비교하였습니다. 각 벤치마크에서의 safety score(%)가 표로 제시되어 있고, 점수가 높을수록 더 안전하다는 의미입니다.
    
    2. **DNA 및 HarmBench 벤치마크**: DNA와 HarmBench는 위험한 지시를 모델이 얼마나 잘 거부하는지를 중심으로 평가합니다. 이 두 벤치마크의 평가는 공식 평가 방법론을 그대로 재현(reproduce)해서 이루어졌고, 그 외의 벤치마크 결과는 HELM 플랫폼에서 가져왔습니다.
    
    3. **LLM-as-a-Judge 방식**: 각 질문에 대한 모델의 응답을 대형 언어 모델(LLM, 주로 GPT-4o 등)이 직접 평가(safe, unsafe, rejected)하여 점수를 매깁니다. 이 방식의 정확도는 샘플링 결과에 근거해 95% 이상의 일치도를 보였습니다.
    
    4. **리스크 컨트롤 시스템(Risk Control System)**: 모델의 기본 버전과 리스크 컨트롤 시스템이 적용된 버전을 따로 평가했으며, 리스크 컨트롤 도입 시 unsafe rate(위험 응답 비율)는 감소하지만 rejection rate(응답 거부 비율)는 증가했습니다. 각 결과는 점수와 unsaf

In [19]:
print(result2.final_output)

이 논문에서 사용된 evaluation results는 주로 모델의 안전성(safety) 평가에 초점을 맞추고 있습니다. 주요 평가 결과는 다음과 같이 요약할 수 있습니다:

1. **외부 벤치마크 사용**: 논문에서는 다양한 안전성 관련 벤치마크(예: SST, BBQ, ART, XSTest, DNA(Do-Not-Answer), HarmBench)를 사용하여 DeepSeek-R1과 다른 최첨단 모델(Claude-3.7-Sonnet, o1, GPT-4o, Qwen2.5 등)의 성능을 비교하였습니다. 각 벤치마크에서의 safety score(%)가 표로 제시되어 있고, 점수가 높을수록 더 안전하다는 의미입니다.

2. **DNA 및 HarmBench 벤치마크**: DNA와 HarmBench는 위험한 지시를 모델이 얼마나 잘 거부하는지를 중심으로 평가합니다. 이 두 벤치마크의 평가는 공식 평가 방법론을 그대로 재현(reproduce)해서 이루어졌고, 그 외의 벤치마크 결과는 HELM 플랫폼에서 가져왔습니다.

3. **LLM-as-a-Judge 방식**: 각 질문에 대한 모델의 응답을 대형 언어 모델(LLM, 주로 GPT-4o 등)이 직접 평가(safe, unsafe, rejected)하여 점수를 매깁니다. 이 방식의 정확도는 샘플링 결과에 근거해 95% 이상의 일치도를 보였습니다.

4. **리스크 컨트롤 시스템(Risk Control System)**: 모델의 기본 버전과 리스크 컨트롤 시스템이 적용된 버전을 따로 평가했으며, 리스크 컨트롤 도입 시 unsafe rate(위험 응답 비율)는 감소하지만 rejection rate(응답 거부 비율)는 증가했습니다. 각 결과는 점수와 unsafe/rejection rate로 보고됩니다.

5. **멀티링구얼 및 Jailbreak 안전성 평가**: 50개의 언어에서 번역된 안전성 테스트셋을 통해 모델의 다국어 안전성도 평가했고, 다양한 jailbreaking 공격(모델의 안전망을 우회하는 시도)에 대한 견고성도 별도로 

In [22]:
# 실행 결과 상세 확인
print("실행 결과 상세:")
print("=" * 50)

for idx, item in enumerate(result2.new_items):
    print(f"\n[Item {idx + 1}]")
    print(f"Tool Name: {type(item).__name__}")
    if hasattr(item, 'raw_item'):
        print(f"Raw Item: {str(item.raw_item)[:500]}") # raw_item이 너무 길 수 있으므로 처음 500자만 출력

실행 결과 상세:

[Item 1]
Tool Name: ToolCallItem
Raw Item: ResponseFileSearchToolCall(id='fs_0b1fb037dbf3c35b0069ead791f1c08196bb30097fd547a8da', queries=['evaluation results', 'What evaluation results are used in the paper?'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='file-5fKN1azY9dgYtmyENn2Xo6', filename='DeepSeeek_OCR_paper.pdf', score=0.7276, text='During the reproduction of the HarmBench\r\nresults, we observe that using relatively smaller models (i.e., LLaMA-2-13B) led to unreliable\r\nevaluation outcome

[Item 2]
Tool Name: MessageOutputItem
Raw Item: ResponseOutputMessage(id='msg_0b1fb037dbf3c35b0069ead79725e88196b139a1632a8a414f', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-5fKN1azY9dgYtmyENn2Xo6', filename='DeepSeeek_OCR_paper.pdf', index=1239, type='file_citation'), AnnotationFileCitation(file_id='file-5fKN1azY9dgYtmyENn2Xo6', filename='DeepSeeek_OCR_paper.pdf', index=1239, type='file_citation'), Annota

# 5. handoff_description

In [ ]:
from agents import Agent, FileSearchTool, Runner, trace

# 1. RAG 전문가 에이전트를 만들고 싶다면?
rag_specialist = Agent(
    name="RAG Specialist",
    handoff_description="Deepseek 논문에 대한 질문을 처리하는 전문가, 논문 내용, 기술적 세부사항, 벤치마크 결과 등 질문에 답변합니다.",
    instructions="""You are a RAG specialist agent focused on answering questions about the Deepseek OCR paper. 
    
    Your role:
    - Search the vector store for relevant information.
    - Ptovide accurate, document-based answers
    - Include specific details, numbers, and quotes when available.
    - Answer in Korean
    """, 
    
    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 5, # 상위 5개
            include_search_results = True
        )
    ]
)

# 2. 일반 어시스턴트 에이전트를 만들고 싶다면?
general_assistant = Agent(
    name="General Assistant",
    handoff_description="일반적인 질문이나 인사, 잡담 등을 처리하는 어시스턴트, 문서와 관련없는 일반적인 대화를 담당합니다.",
    instructions="""You are a friendly general assistant.

    Your role:
    - Handle greetings and casual conversations
    - Answer general knowledge questions
    - Provide helpful responses for non-document queries
    - Be friendly and conversational
    - Answer in Korean
    """, 
)

# 3. 요약 전문가 에이전트를 만들고 싶다면?
summarizer = Agent(
    name="Summarizer",
    handoff_description="논문의 전체 요약이나 특정 섹션 요약을 요청할 때 사용하는 요약 전문가", 
    instructions="""You are a summarization expert for the DeepSeek OCR paper.

    Your role:
    - Search the document and create comprehensive summaries
    - Provide structured summaries with key points
    - Highlight important findings and contributions
    - Use bullet points for clarity
    - Answer in Korean
    """, 

    tools = [
        FileSearchTool(
            vector_store_ids = [vector_store.id],
            max_num_results = 10, # 요약을 위해 더 많은 결과
            include_search_results = True
        )
    ]
)

# 6. Triage Agent 

In [12]:
triage_agent = Agent(
    name="Triage Agent",
    instructions="""You are a triage agent that routes user questions to the appropriate specialist.

    Routing rules:
    1. ** RAG Specialist **: Questions about the DeepSeek OCR paper conten , technical details, methodology, experiments, or results
    2. ** Summarizer **: Requests for summaries, overviews, or key takeaways
    From the paper
    3. ** General Assistant **: Greetings, general questions, or anything not related to the document

    Always analyze the user's intent carefully before routing.
    Do NOT answer questions directly - always hand off to the appropriate specialist.
    """,
    handoffs=[rag_specialist, summarizer, general_assistant]
)

print(f"Trigger Agent 생성 완료!")
print(f"handsoffs: {[agent.name for agent in triage_agent.handoffs]}")

Trigger Agent 생성 완료!
handsoffs: ['RAG Specialist', 'Summarizer', 'General Assistant']


# 7. 멀티 에이전트 RAG 실행

In [13]:
async def ask_triage_agent(question: str):
    """트리아지 에이전트에게 질문하기"""
    with trace("Multi-Agent RAG"):
        result = await Runner.run(triage_agent, question)
        return result
    
    
# 테스트 1: 일반 인사 (-> General Assistant)
test1 = "안녕하세요. 오늘 기분이 어때요?"

print(f"질문: {test1}")
print("=" * 50)

result = await ask_triage_agent(test1)
print(f"답변: {result.final_output}")
print(f"최종 에이전트: {result.last_agent.name}")

질문: 안녕하세요. 오늘 기분이 어때요?


Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


답변: 안녕하세요! 저는 항상 기분이 좋아요. 오늘도 여러분을 도와드릴 준비가 되어 있습니다. 😊  
혹시 궁금한 점이나 필요한 게 있으시면 언제든 말씀해 주세요!
최종 에이전트: General Assistant


In [16]:
# 테스트 2: 논문 내용 질문 (-> RAG Specialist)
test2 = "DeepSeek OCR 논문의 제안된 Architecture가 무엇인지 간단히 설명해줘."

print(f"질문: {test2}")
print("=" * 50)

result = await ask_triage_agent(test2)
print(f"답변: {result.final_output}")
print(f"최종 에이전트: {result.last_agent.name}")

Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


질문: DeepSeek OCR 논문의 제안된 Architecture가 무엇인지 간단히 설명해줘.
답변: 파일에서 직접적으로 "제안된 Architecture"에 대한 언급을 찾을 수는 없었습니다. 하지만 논문의 주요 내용을 바탕으로 DeepSeek OCR의 제안된 아키텍처에 대해 간단히 설명드릴 수 있습니다.

DeepSeek OCR 논문이 제안한 아키텍처는 최근 LLM(대형 언어 모델)과 Vision 모델을 결합한 멀티모달 구조를 바탕으로 하고 있습니다. OCR(Optical Character Recognition)의 인식 성능을 극대화하기 위해, 이미지의 시각적 피처와 텍스트 정보를 동시에 받아들이고 이해할 수 있는 구조로 설계되었습니다. 

구체적으로는 시각 피처를 추출하는 Vision Backbone(예: CNN 또는 Transformer 구조)과, 텍스트 처리를 위한 Language Model을 결합하고, 멀티모달 정보를 효과적으로 융합하는 Cross Attention 등의 기법을 활용합니다. 이후 최종적으로 텍스트 인식 결과를 출력하는 구조로 이루어져 있습니다.

조금 더 구체적인 구성도나 표가 필요하다면, 논문 내 해당 부분(Introduction, Method 등)을 참고하여 직접 요약해드릴 수 있습니다. 원하시면 구체적인 부분을 명시해주시면 더 자세히 찾아드리겠습니다!
최종 에이전트: RAG Specialist


In [18]:
# 테스트 3: 요약 요청 (-> Summarizer)
test3 = "DeepSeek의 논문을 요약해줘."

print(f"질문: {test3}")
print("=" * 50)

result = await ask_triage_agent(test3)
print(f"답변: {result.final_output}")
print(f"최종 에이전트: {result.last_agent.name}")

Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_RAG Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_rag_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_General Assistant' contains invalid characters for function calling and has been transformed to 'transfer_to_general_assistant'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


질문: DeepSeek의 논문을 요약해줘.
답변: 검색 키워드로는 관련 내용을 찾을 수 없었습니다. 논문 내용을 살펴본 후 핵심적인 내용을 요약해드리겠습니다. 잠시만 기다려 주세요. 

전체적으로 논문의 목표, 구조, 주요 실험, 그리고 기여점 등을 중심으로 요약해드리겠습니다.  
(필요하면 추가적으로 논문 내의 표, 그림, 결과 등을 구체적으로 찾아 반영할 수 있습니다.) 

논문 파일 내 주요 내용을 직접 확인하려면, 구체적으로 "DeepSeek의 목적", "모델 아키텍처", "실험 결과", "기여점" 등으로 추가로 질문하거나, 논문 첫 부분(초록/서론)부터 차례로 요약해드릴 수 있습니다.  
원하시는 방향이 있으시면 말씀해 주세요!  

우선, 전체 논문의 핵심을 요약해서 전달드리겠습니다.  잠시만 기다려 주세요.
최종 에이전트: Summarizer


# 리소스 정리

In [20]:
cleanup = input("Vector Store을 삭제하시겠습니까? (y/n): ")
if cleanup.lower() == 'y':
    client.vector_stores.delete(vector_store.id)
    print("Vector Store이 삭제되었습니다.")

print(f"현재 Vector Store ID: {vector_store.id}")

Vector Store이 삭제되었습니다.
현재 Vector Store ID: vs_69eafd01e3988191b0f3e2c5ca3f2897


In [21]:
try:
    client.vector_stores.retrieve(vector_store.id)
except Exception as e:
    print("이미 삭제됨")

이미 삭제됨
